# Generate matched AI gift designs
Run on a Colab **GPU** runtime (T4 is enough). Produces `ai01.png` ... `ai30.png` with fixed seeds, so the run can be repeated exactly.

In [1]:
!pip install -q diffusers transformers accelerate

In [2]:
import torch
assert torch.cuda.is_available(), 'Switch the runtime to a GPU first'
print(torch.cuda.get_device_name(0))

Tesla T4


In [3]:
PROMPTS = [
    ('01', 'Handmade birthday gift basket: white woven rectangular basket trimmed with red satin ribbon, cartoon superhero and balloon stickers on the front, filled with folded burgundy paisley fabric, a purple candy jar, a chocolate bar, a pink packet of instant spicy noodles, a bag of cheese puffs, a cut-out letter Happy Birthday card and a card of stud earrings, covered with white tulle gathered at the top and tied with a large red satin bow, on a side table in a living room with a floor lamp and a potted plant, product photograph'),
    ('02', "Clear acrylic square gift box seen from above, filled with women's accessories: a gold necklace and earring set in a black presentation box, a press-on nail kit, gold bangles, a pink flower hair claw, satin scrunchies, a small round mirror, a keychain on a yellow card, a perfume bottle and chocolates, string fairy lights woven through, dim indoor light on a dark floor, product photograph"),
    ('03', 'White woven gift basket with a handle, holding a white plush cat with blue ears, red roses, a red bottle, a small jar of candies and snack packets in red packaging, wrapped in white tulle with a red ribbon pull-bow on top, warm fairy lights inside, surrounded by red roses, warm evening light, product photograph'),
    ('04', 'Small white woven rectangular basket with a red satin band and gold trim, artificial red roses and green leaves on top, a pink skincare tube, a pink roll-on bottle, chocolate bars, a caramel snack box, a pink satin scrunchie, red satin bows and a tulle-wrapped item, on a dark carved wooden coffee table in a living room with a soft blurred background, product photograph'),
    ('05', 'Chocolate bouquet in a round black box tied with a red satin ribbon, red and white foam roses, chocolate bars in purple and yellow wrappers standing upright, a handwritten note card, outdoors on dry grass in a park in sunlight, product photograph'),
    ('06', 'Get-well fruit basket: white woven rectangular basket with a white handle, filled with apples, loquats, guavas, a melon and bananas, a heart-shaped handmade Get Well Soon card with blue lettering and small hearts tied with blue satin ribbon, covered with white tulle gathered at the top with a blue satin bow, on a round side table in a living room by a window, product photograph'),
    ('07', "Three small pale pink paper gift bags with handles held up by hands, each holding small cartoon-animal plush toys and dried baby's breath in red, pink and beige, tied with red and white string, bright room, product photograph"),
    ('08', 'Three crocheted sunflower bouquets held up by hands, wrapped in yellow, pink and blue paper, each with a yarn sunflower, pipe-cleaner leaves, a small plush duck or penguin and a card reading sunshine, bright room, product photograph'),
    ('09', 'Three plush toy bouquets held up by hands, wrapped in pink, blue and yellow paper, each holding several small plush animals such as bunnies, ducks, a panda and a teddy bear, tied with ribbons, bright room, product photograph'),
    ('10', 'Makeup bouquet held in one hand, wrapped in hot pink paper and pastel heart-print wrapping, tied with small pink bows, holding skincare and makeup products in pink packaging: a face mask, a liquid blush tube, a serum box, an eyeshadow palette and a lip tint, plain wall background, product photograph'),
    ('11', "Round white woven basket held in hand, blue satin ribbon band and bows, a white teddy bear, an acrylic butterfly topper, chocolate bars, baby's breath and items wrapped in white tulle tied with blue ribbon, in front of a green plant, product photograph"),
    ('12', 'Open brown cardboard mailer box birthday gift, inside lid decorated with a Happy Birthday pennant banner, pink paper cut-out stickers and fairy lights, inside the box pink snack packets, a pink instant noodle pack, a small brown teddy bear, a pearl hair claw and a handwritten letter, on a white chair by a window, product photograph'),
    ('13', 'Wedding gift basket with a gold and white twisted handle, filled with red sequined fabric, a decorated tray with pink paper flowers, red paper fans and fairy lights, covered with red and white tulle, gold organza ribbon and a white bow, indoor living room, product photograph'),
    ('14', "Men's gift basket in a gold and white woven tray with a blue and gold twisted handle, containing a deodorant spray, a men's face wash tube, a chocolate bar, a black perfume box, white artificial carnations and an Arabic calligraphy card, outdoors on grass, product photograph"),
    ('15', 'Large clear acrylic gift box tied with sparkly black ribbon, fairy lights inside, filled with chocolates, a small clutch purse with gold initial letters, gold decorative pieces and a black box, on a cream rug beside a vase of orange and white flowers and a green handbag, product photograph'),
    ('16', 'Two matching black gift boxes lined with white tissue paper, each holding a handmade printed music-player card with heart artwork, butterfly cut-outs, sticky notes, a notebook, rolled paper scrolls and small treats, behind them two pink framed cards with red quilled paper hearts and handwritten names, beside artificial white and burgundy flowers, a light blue teddy bear and a crystal bowl of red paper confetti, on a table indoors, product photograph'),
    ('17', 'White woven rectangular basket with a pink fabric rim, filled with snack packets in teal and dark packaging and two cups of spicy snacks, covered with bright pink tulle gathered into a large puff at the top, indoor living room in the evening, product photograph'),
    ('18', 'Set of translucent pink glass bangles with gold glitter edges and small gold jingle bells, held in an open palm, carpeted room in the background, product photograph'),
    ('19', 'Clear gift box with gold trim holding a pink ceramic mug with small hearts, tied with a peach satin ribbon printed with gold hearts and the word love, beside dried purple flowers, on a wooden table with a glass top, product photograph'),
    ('20', 'Pair of gold kundan jhumka earrings with mirror stones and pearl drops, mounted on a handmade card with a red paisley border and the word Jhumka in teal letters, topped with a red satin bow printed with gold Arabic script, held in hand, product photograph'),
    ('21', 'Round gold woven Eid gift basket with a red satin band printed with gold Eid greetings in Arabic script, holding red flowers, Arabic calligraphy greeting cards and gold kundan earrings on a red card, covered with white tulle tied with a red satin bow, on a wooden side table by a window next to a white vase of pink artificial roses, product photograph'),
    ('22', 'Two white square cushions printed with a cute cartoon panda and a brown bear hugging, with red hearts, on a sofa with dark and sequined cushions, product photograph'),
    ('23', 'Handmade heart-shaped bouquet of white tissue-paper flowers with warm fairy lights, wrapped in lime green paper with white polka dots, tied with a red satin ribbon, seen from above, product photograph'),
    ('24', 'White cardboard gift box with a clear window and a silver geometric pattern, holding pastel candles, tied with a red satin ribbon bow, on a wooden side table with gold decor and a small potted plant, product photograph'),
    ('25', 'Handmade heart-shaped Get Well Soon card with blue hand-lettering and small doodled hearts, tied with a blue satin ribbon, held up in hand in front of a sunlit window, product photograph'),
    ('26', 'Small white woven birthday basket with a white handle tied with two red satin bows, colorful Happy Birthday pennant banners, a small white teddy bear, stud earring cards, a pink instant noodle packet, a hand cream and a chocolate box, on a dark wooden cabinet, product photograph'),
    ('27', 'Frosted plastic gift bag holding a small cream teddy bear with pink roses, a purple satin scrunchie and a pink candle jar, with an orange ring keychain tied on with a purple satin ribbon, beside a green plant, warm light, product photograph'),
    ('28', 'Small hand-held bouquet of navy blue satin ribbon roses with pearl centers, wrapped in black and silver snowflake-print paper, tied with a navy satin bow, against a plain white wall, product photograph'),
    ('29', "Chocolate drip cake with rainbow sprinkles and an upside-down waffle cone on a gold board, on a wooden tray lined with white paper, framed by red roses, baby's breath and greenery, with fairy lights and red ribbons, product photograph"),
    ('30', 'Large gold woven basket wrapped in purple tulle with a purple pull bow, holding two cushions printed with a cute cartoon panda and bear, a plush toy in a blue hat, a clear box with a red rose and wafer rolls, a white water bottle and a checked scarf, on a brown carpet beside gold balloons, product photograph'),
]

# Put the ids of photos you dropped here, e.g. {'18'}
SKIP = set()


In [4]:
from diffusers import StableDiffusionXLPipeline

MODEL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
STEPS = 30
GUIDANCE = 7.0
BASE_SEED = 1000

pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, variant='fp16', use_safetensors=True).to('cuda')

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [ ]:
import os
os.makedirs('ai', exist_ok=True)
for pid, prompt in PROMPTS:
    if pid in SKIP:
        continue
    g = torch.Generator('cuda').manual_seed(BASE_SEED + int(pid))
    img = pipe(prompt=prompt, num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=g).images[0]
    img.save(f'ai/ai{pid}.png')
    print('saved', pid)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (106 > 77). Running this sequence through the model will result in indexing errors
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer CLIPTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['at the top and tied with a large red satin bow , on a side table in a living room with a floor lamp and a potted plant , product photograph']
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (106 > 77). Running this sequ

  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:750: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['indoor light on a dark floor , product photograph']
The following part of you

saved 01


  0%|          | 0/30 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


... (19 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


saved 09


  0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
import json
json.dump({'model': MODEL_ID, 'steps': STEPS, 'guidance': GUIDANCE, 'seed_rule': 'BASE_SEED + id', 'base_seed': BASE_SEED, 'skipped': sorted(SKIP)}, open('ai/generation_settings.json', 'w'), indent=2)
!zip -qr ai_images.zip ai
print('zipped')

: 

## Get the zip onto your laptop
In the browser version of Colab, the next cell downloads it. If you run this from VS Code and the download does not appear, open the same notebook at colab.research.google.com and run it there.

In [ ]:
from google.colab import files
files.download('ai_images.zip')